#### InteractiveMarker wrist IK -> real `FFW SG2`

Drag the two gizmos (one per wrist) with the mouse to move the left/right arm targets.
Shift-click a gizmo to switch it to rotate mode. Both wrist targets go into one
collision-aware IK solve, and the resulting joint command is published to the real SG2.

Run order:
1. `real_render_MetaQuest3_sg2.ipynb` in another kernel/window (optional, shows real vs command).
2. This notebook. `Robot command` starts at `OFF`; switch it to `ON` only after the gizmos sit on the wrists.
3. `Marker` -> `RESET` snaps both gizmos back onto the current SG2 wrists, then returns to `TRACK`.


**셀 1. 프로젝트 초기화 및 모듈 임포트**

프로젝트 초기화 스크립트(`init_project.py`)를 실행하고, 시뮬레이션/키네마틱스/유틸리티/Qt 관련 패키지들을 임포트한다. 마지막으로 Qt 애플리케이션 인스턴스(`app`)를 생성한다.

In [1]:
%run ../package/init_project.py

import numpy as np

from ri_motion_v5_package.init_scripts.init_ipython_setup import *
from ri_motion_v5_package.init_scripts.init_qt import *
from ri_motion_v5_package.mujoco_sim import *
from ri_motion_v5_package.kinematics import *
from ri_motion_v5_package.utility import *
from ri_motion_v5_package.qt import *
from utils import *

app = get_qt_app()


[ffw_sg2_vr_teleoperation] initialized.
 - PROJECT_DIR: /Users/jangjiyun/Desktop/JY/code/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation
Basic packages initialized.
 import numpy as np
 import matplotlib.pyplot as plt
 import scipy as sp
 from PIL import Image
[ipython] settings (retina, inline, qt) initialized.
[numpy] settings initialized.
[matplotlib] settings initialized.
[qt] package initialized.


**셀 2. 충돌 인식(collision-aware) IK 네이티브 백엔드 빌드**

SG2용 충돌 인식 IK를 수행하는 네이티브 백엔드를 빌드(또는 로드)한다. `required=True`이므로 빌드에 실패하면 예외가 발생한다. 빌드가 끝나면 바인딩 정보를 출력해 정상적으로 로드되었는지 확인한다.

In [2]:
native_backend = build_sg2_collision_aware_ik_backend(
    clean   = False,
    verbose = True,
    required = True,)
print("bind info:",native_backend.native_info())

[build_native]
  module      : _mujoco_collision_aware_ik_bind
  output      : mujoco_sim/so/_mujoco_collision_aware_ik_bind.cpython-310-darwin.so
  compiler    : clang++
  project cpp:
    [01] mujoco_sim/cpp/bindings.cpp
    [02] mujoco_sim/cpp/signed_distance_bindings.cpp
    [03] mujoco_sim/cpp/signed_distance.cpp
    [04] mujoco_sim/cpp/ik_bindings.cpp
    [05] mujoco_sim/cpp/ik.cpp
  project hpp:
    [01] mujoco_sim/cpp/signed_distance_bindings.hpp
    [02] mujoco_sim/cpp/signed_distance.hpp
    [03] mujoco_sim/cpp/ik_bindings.hpp
    [04] mujoco_sim/cpp/ik.hpp
    [05] mujoco_sim/cpp/mujoco_handles.hpp
  defines:
    [01] RI_NATIVE_ENABLE_SIGNED_DISTANCE
    [02] RI_NATIVE_ENABLE_IK
    [03] RI_NATIVE_MODULE_NAME=_mujoco_collision_aware_ik_bind
  nanobind cpp:
    [01] nanobind/src/nb_combined.cpp
  commands    : hidden; pass show_commands=True to print full commands
[build_native] compile
  [01/06] mujoco_sim/cpp/bindings.cpp -> build/native/bindings.o
  [02/06] mujoco_sim/cpp/

**셀 3. 통신(ZMQ) 포트 및 IP 설정**

렌더러로 상태를 퍼블리시할 포트(`RENDER_PORT`), 실제 SG2 로봇으로 명령을 퍼블리시할 포트/IP(`PUB_TO_SG2_PORT`, `PUB_TO_SG2_IP`), 그리고 SG2 로봇의 현재 상태를 구독할 포트/IP(`SUB_FROM_SG2_PORT`, `SUB_FROM_SG2_IP`)를 정의한다. `ALLOW_MISSING_ROBOT_STATE`는 로봇 상태 수신이 안 될 때 예외를 허용할지 여부를 결정한다.

In [3]:
RENDER_PORT = 5556
PUB_TO_SG2_PORT = 5561
PUB_TO_SG2_IP = get_ip_address()
SUB_FROM_SG2_PORT = 5560
SUB_FROM_SG2_IP = "192.168.6.2"

ALLOW_MISSING_ROBOT_STATE = False

print("PUB_TO_SG2_IP",PUB_TO_SG2_IP)
print("Robot state SUB: tcp://%s:%d"%(SUB_FROM_SG2_IP,SUB_FROM_SG2_PORT))

PUB_TO_SG2_IP 192.168.6.103
Robot state SUB: tcp://192.168.6.2:5560


**셀 4. MuJoCo 시뮬레이션 환경 로드**

바닥(`FLOOR_XML_PATH`)과 SG2 로봇(`SG2_XML_PATH`) MJCF 파일을 병합해 하나의 씬 XML(`SG2_SCENE_XML_PATH`)을 만들고, 이를 이용해 `MuJoCoParser` 환경(`env`)을 생성한다.

In [4]:
xml_path = merge_mjcfs(
    included_mjcf_files = [
        str(FLOOR_XML_PATH),
        str(SG2_XML_PATH),
    ],
    output_xml_path = str(SG2_SCENE_XML_PATH),)
env = MuJoCoParser(rel_xml_path=xml_path,verbose=False)

[merge_mjcfs] Merging [2] MJCF files:
  - [0] [/Users/jangjiyun/Desktop/JY/code/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/floor_simple_white.xml]
  - [1] [/Users/jangjiyun/Desktop/JY/code/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/ffw_sg2.xml]
[merge_mjcfs] Saved merged XML to:[/Users/jangjiyun/Desktop/JY/code/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml]
MuJoCo model loaded from [/Users/jangjiyun/Desktop/JY/code/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].


**셀 5. 관절 메시지 레이아웃 구성 및 실제 로봇 초기 자세 동기화**

환경을 리셋하고 베이스 위치를 세팅한 뒤, 실제 로봇으로 보낼 명령 관절 이름 목록(`command_joint_names`)과 각 관절 그룹(lift/head/arm_l/arm_r/gripper_l/gripper_r)을 메시지 필드로 매핑하는 `q_msg_layout`을 만든다. 관절 최소/최대값도 조회한다. 이후 SG2 로봇으로부터 현재 관절 상태를 구독(`get_latest_robot_qpos`)해 시뮬레이션 자세를 실제 로봇 자세로 맞춰서 시작 시 점프가 생기지 않도록 한다(수신 실패 시 `ALLOW_MISSING_ROBOT_STATE`에 따라 예외를 던지거나 현재 MuJoCo 자세로 대체).

In [5]:
env.reset()
env.set_p("base_link","body",(0.0,0.0,0.01))

real_command_joint_names = (
    list(SG2_REAL_COMMAND_JOINT_NAMES)
    + [SG2_GRIPPER_DRIVER_JOINT_NAMES[hand_type] for hand_type in SG2_HAND_TYPES]
)
command_joint_names = filter_existing_joint_names(env,real_command_joint_names)
q_msg_format_dict = {
    "lift"      : {"prefixes":["lift_joint"]},
    "head"      : {"prefixes":["head_joint"]},
    "arm_l"     : {"prefixes":["arm_l_joint"]},
    "arm_r"     : {"prefixes":["arm_r_joint"]},
    "gripper_l" : {"prefixes":["gripper_l_joint1"]},
    "gripper_r" : {"prefixes":["gripper_r_joint1"]},
}
q_msg_layout = build_q_msg_layout(
    env               = env,
    q_msg_format_dict = q_msg_format_dict,
    use_joint_names   = command_joint_names,)
q_use_min = env.get_joint_mins(command_joint_names)
q_use_max = env.get_joint_maxs(command_joint_names)

print("Joint message layout")
print("--------------------")
for q_msg_key in q_msg_layout["q_msg_keys"]:
    joint_names = q_msg_layout["qmsgkeys_to_usejointnames_dict"][q_msg_key]
    joint_idxs = q_msg_layout["qmsgkeys_to_usejointidxs_dict"][q_msg_key]
    print("%-8s | %2d joints | idxs:%s | %s"%(q_msg_key,len(joint_names),joint_idxs,joint_names))
print("--------------------")
print("%-8s | %2d joints"%("total",len(command_joint_names)))
print("unused:",q_msg_layout["unused_joint_names"])

try:
    q_use,recv_count = get_latest_robot_qpos(
        env             = env,
        q_msg_layout    = q_msg_layout,
        use_joint_names = command_joint_names,
        sub_ip          = SUB_FROM_SG2_IP,
        sub_port        = SUB_FROM_SG2_PORT,
        timeout_sec     = 1.0,
        verbose         = True,)
    print("[Robot init] OK. recv_count:[%d]"%recv_count)
except Exception:
    if not ALLOW_MISSING_ROBOT_STATE:
        raise
    q_use = env.get_qpos(joint_names=command_joint_names)
    print("[Robot init] missing robot state; using current MuJoCo q because ALLOW_MISSING_ROBOT_STATE=True.")

# Start the sim at the real robot pose so the gizmos seed without a jump.
q_use = np.clip(q_use,q_use_min,q_use_max)
env.forward(q=q_use,joint_names=command_joint_names)

Joint message layout
--------------------
lift     |  1 joints | idxs:[0] | ['lift_joint']
head     |  2 joints | idxs:[1, 2] | ['head_joint1', 'head_joint2']
arm_l    |  7 joints | idxs:[3, 4, 5, 6, 7, 8, 9] | ['arm_l_joint1', 'arm_l_joint2', 'arm_l_joint3', 'arm_l_joint4', 'arm_l_joint5', 'arm_l_joint6', 'arm_l_joint7']
arm_r    |  7 joints | idxs:[10, 11, 12, 13, 14, 15, 16] | ['arm_r_joint1', 'arm_r_joint2', 'arm_r_joint3', 'arm_r_joint4', 'arm_r_joint5', 'arm_r_joint6', 'arm_r_joint7']
gripper_l |  1 joints | idxs:[17] | ['gripper_l_joint1']
gripper_r |  1 joints | idxs:[18] | ['gripper_r_joint1']
--------------------
total    | 19 joints
unused: []
[ZMQ Subscriber] pid:[40600] endpoint:[tcp://192.168.6.2:5560]
[ZMQ] role:[sub] ip:[192.168.6.2] port:[5560] hwm:[1] conflate:[True]
[ZMQ SUB] pid:[40600] Socket closed.
[Robot init] OK. recv_count:[53]


**셀 6. IK 솔버, 뷰어, 인터랙티브 마커, UI 위젯, ZMQ 퍼블리셔 초기화**

루프 주기(input/robot/render Hz)를 정의하고, SG2 자기충돌 검사용 지오메트리 쌍(`collision_geom_pairs`)과 양팔용 충돌 인식 IK 솔버(`collision_ik_solvers`)를 생성한다. 뷰어를 열고 카메라를 세팅한 뒤, 좌/우 손목 각각에 대해 드래그 가능한 `InteractiveMarker` 기즈모를 실제 로봇의 현재 손목 위치에 맞춰 생성한다. 로봇 명령 ON/OFF 및 마커 TRACK/RESET을 제어하는 라디오 버튼 위젯(`ctrl_radio`)과 그리퍼 개폐용 슬라이더 위젯(`gripper_slider`)을 만든다. 마지막으로 실제 로봇으로 명령을 보낼 퍼블리셔(`pub_robot`)와 렌더 상태를 보낼 퍼블리셔(`pub_render`)를 생성한다.

In [6]:
input_hz = 30
robot_hz = 100
render_hz = 20

hand_types = list(SG2_HAND_TYPES)
hand_labels = dict(SG2_HAND_LABELS)

collision_geom_pairs,collision_candidate_info = create_sg2_collision_candidate_geom_pairs(
    env               = env,
    init_exclude_dist = 0.05,)
print_sg2_collision_candidate_summary(collision_candidate_info)

collision_ik_solvers,arm_joint_names = create_sg2_collision_aware_arm_ik_solvers(
    env                = env,
    mujoco_parser_cls  = MuJoCoParser,
    body_ik_solver_cls = BodyPositionIKSolverRevPriBase,
    enable_depenetration = False,)
active_ik_solver = collision_ik_solvers["tracking"]
collision_ik_result = make_idle_collision_ik_result(active_ik_solver,safe_margin=0.05)

last_solved_wrist_target = {hand_type:None for hand_type in hand_types}
arm_ik_info_by_hand = {hand_type:None for hand_type in hand_types}
T_sg2_wrist = {hand_type:get_sg2_wrist_control_T(env,hand_type) for hand_type in hand_types}
render_arm_ik_targets = []

gripper_driver_joint_names = [SG2_GRIPPER_DRIVER_JOINT_NAMES[hand_type] for hand_type in hand_types]
gripper_q_by_hand = {
    hand_type:float(env.get_qpos(joint_names=[SG2_GRIPPER_DRIVER_JOINT_NAMES[hand_type]])[0])
    for hand_type in hand_types
}

env.init_viewer(title="SG2 Marker IK -> Real FFW SG2",x_offset=0.24,y_offset=0.00,width=0.76,height=1.0,fontscale=180)
env.viewer.set_cam_info(180.0,2.8,-20.0,[0.0,0.0,0.95])
env.viewer.set_transparency(transparent=True)

# One draggable gizmo per wrist, seeded at the real robot's current wrist pose.
markers = {
    hand_type: InteractiveMarker(
        env          = env,
        T            = get_sg2_wrist_control_T(env,hand_type),
        axis_len     = 0.12,
        ring_r       = 0.12,
        control_mode = "pR",)
    for hand_type in hand_types
}

ctrl_radio = MultiRadioQtWidget(
    title          = "SG2 Marker Command",
    x_offset       = 0.00,
    y_offset       = 0.00,
    window_width   = 0.24,
    window_height  = 0.14,
    label_texts    = ["Robot command","Marker"],
    option_texts   = [["OFF","ON"],["TRACK","RESET"]],
    initial_texts  = ["OFF","TRACK"],)

gripper_slider = MultiSliderQtWidget(
    title          = "SG2 Gripper",
    x_offset       = 0.00,
    y_offset       = 0.18,
    window_width   = 0.24,
    window_height  = 0.14,
    label_texts    = [SG2_GRIPPER_DRIVER_JOINT_NAMES[hand_type] for hand_type in hand_types],
    slider_mins    = env.get_joint_mins(gripper_driver_joint_names),
    slider_maxs    = env.get_joint_maxs(gripper_driver_joint_names),
    slider_vals    = [gripper_q_by_hand[hand_type] for hand_type in hand_types],
    resolutions    = [0.01]*len(hand_types),)

pub_robot = ZMQPubSub(role="pub",port=PUB_TO_SG2_PORT,ip=PUB_TO_SG2_IP,verbose=True)
pub_render = ZMQPubSub(role="pub",port=RENDER_PORT,verbose=True)

print("ZMQ publishers")
print("--------------")
print("Robot    : tcp://*:%d"%PUB_TO_SG2_PORT)
print("Renderer pub : tcp://*:%d"%RENDER_PORT)
print("Render view  : run real_render_MetaQuest3_sg2.ipynb in another kernel/window")
print("collision IK joints:",active_ik_solver.joints_use)
print("Depenetration IK: disabled.")
print("Input: mouse gizmos. Shift-click a gizmo to rotate it.")
print("Robot command starts OFF. Turn it ON only after checking the gizmo poses.")

Collision candidates
--------------------
robot collision geom:[34]
collision geom:[35]
candidate geom pairs:[493]
excluded initial geom pairs:[37]
MuJoCo model loaded from [/Users/jangjiyun/Desktop/JY/code/VAI_AIWORKER/ri_motion_v5_VR/project/ffw_sg2_vr_teleoperation/xml/scene_ffw_sg2_metaquest3.xml].
[ZMQ Publisher] pid:[40600] endpoint:[tcp://0.0.0.0:5561]
[ZMQ] role:[pub] ip:[192.168.6.103] port:[5561] hwm:[1] conflate:[True]
[ZMQ Publisher] pid:[40600] endpoint:[tcp://0.0.0.0:5556]
[ZMQ] role:[pub] ip:[192.168.6.103] port:[5556] hwm:[1] conflate:[True]
ZMQ publishers
--------------
Robot    : tcp://*:5561
Renderer pub : tcp://*:5556
Render view  : run real_render_MetaQuest3_sg2.ipynb in another kernel/window
collision IK joints: ['arm_l_joint1', 'arm_l_joint2', 'arm_l_joint3', 'arm_l_joint4', 'arm_l_joint5', 'arm_l_joint6', 'arm_l_joint7', 'arm_r_joint1', 'arm_r_joint2', 'arm_r_joint3', 'arm_r_joint4', 'arm_r_joint5', 'arm_r_joint6', 'arm_r_joint7']
Depenetration IK: disabled.
Inp

**셀 7. 메인 루프 — 마커 입력 → IK 계산 → 로봇 명령 퍼블리시 → 렌더링**

뷰어가 살아있고 UI 위젯이 보이는 동안 반복 실행되는 메인 루프다.
- `Robot command` 라디오가 ON/OFF로 바뀌는지 감지해 로그를 출력한다.
- `Marker`가 RESET으로 바뀌면 두 기즈모를 현재 SG2 손목 위치로 스냅시키고 다시 TRACK으로 되돌린다.
- `input_hz` 주기(`tmr_input`)마다: 두 기즈모의 드래그 상태(목표 pose)를 읽어와 `solve_sg2_collision_aware_wrist_ik`로 충돌을 고려한 양팔 IK를 한 번에 풀고, 그리퍼 슬라이더 값을 그리퍼 관절에 반영한 뒤, 관절값을 클리핑해 시뮬레이션에 반영한다.
- `robot_hz` 주기(`tmr_robot`)마다: `Robot command`가 ON이면 현재 관절값을 메시지로 변환해 실제 SG2 로봇으로 퍼블리시한다(OFF면 보내지 않음).
- `render_hz` 주기(`tmr_render`)마다: 좌표축/링크/마커를 그리고 명령 상태·마커 상태·IK/충돌 정보·그리퍼 값을 오버레이 텍스트로 표시한 뒤 뷰어를 렌더링하고, 동일 정보를 렌더 퍼블리셔로 전송해 `real_render_MetaQuest3_sg2.ipynb`에서 시각화할 수 있게 한다.
- 종료 시(예외/Ctrl-C 포함) 모든 퍼블리셔와 위젯을 정리하고 뷰어를 닫는다.

In [7]:
tmr_input = SimpleTimer(name="Marker IK",Hz=input_hz,verbose=True)
tmr_robot = SimpleTimer(name="Robot Pub",Hz=robot_hz,verbose=True)
tmr_render = SimpleTimer(name="Render Pub",Hz=render_hz,verbose=True)
tmr_input.start()
tmr_robot.start()
tmr_render.start()
env.reset_wall_time(); env.reset_sim_time()

robot_pub_hz_actual = 0.0
prev_on_flag = False
wrist_targets_by_hand = {hand_type:marker.get_T(update=False,draw=False) for hand_type,marker in markers.items()}

try:
    while env.is_viewer_alive() and ctrl_radio.isVisible():
        app.processEvents()

        on_flag = ctrl_radio.get_button_option(label_text="Robot command") == "ON"
        if on_flag != prev_on_flag:
            print("[Robot Pub] %s"%("ON" if on_flag else "OFF"))
            prev_on_flag = on_flag

        # Snap both gizmos back onto the current SG2 wrists, then fall back to TRACK.
        if ctrl_radio.get_button_option(label_text="Marker") == "RESET":
            for hand_type,marker in markers.items():
                marker.set_T(get_sg2_wrist_control_T(env,hand_type),update_T0=True)
            ctrl_radio.set_button_option(label_text="Marker",option_text="TRACK")
            print("[Marker] reset to the current SG2 wrists.")

        if tmr_input.do_run():
            # Poll each gizmo's drag/hover state (no draw yet) so the IK solve below sees the latest drag.
            wrist_targets_by_hand = {
                hand_type: marker.get_T(update=True,draw=False)
                for hand_type,marker in markers.items()
            }

            # Both wrist targets in one collision-aware arm IK solve
            arm_ik_info_by_hand = {hand_type:None for hand_type in hand_types}
            collision_ik_result = solve_sg2_collision_aware_wrist_ik(
                env                    = env,
                tracking_solver        = collision_ik_solvers["tracking"],
                depenetration_solver   = None,
                wrist_targets_by_hand  = wrist_targets_by_hand,
                collision_geom_pairs   = collision_geom_pairs,
                native_backend         = native_backend,
                safe_margin            = 0.05,
                signed_distance_topk   = 5,
                enable_depenetration   = False,)
            active_ik_solver = collision_ik_result["solver"]
            last_solved_wrist_target,arm_ik_info_by_hand = update_last_solved_wrist_targets(
                last_solved_wrist_target = last_solved_wrist_target,
                arm_ik_info_by_hand      = arm_ik_info_by_hand,
                attached_wrist_targets   = wrist_targets_by_hand,
                ik_info                  = collision_ik_result["ik_info"],)

            # Gripper sliders -> SG2 gripper joints
            for hand_type,gripper_q in zip(hand_types,gripper_slider.get_values()):
                gripper_q_by_hand[hand_type] = float(gripper_q)
                set_sg2_gripper_qpos(env,hand_type,gripper_q_by_hand[hand_type])

            q_use = np.clip(env.get_qpos(joint_names=command_joint_names),q_use_min,q_use_max)
            env.forward(q=q_use,joint_names=command_joint_names)

            T_sg2_wrist = {
                hand_type:get_sg2_wrist_control_T(env,hand_type)
                for hand_type in hand_types
            }
            render_arm_ik_targets = get_solver_target_infos(active_ik_solver,env)
            tmr_input.end()

        if tmr_robot.do_run():
            if on_flag:
                robot_pub_hz_actual = 1.0/tmr_robot.sec_elps_diff if tmr_robot.sec_elps_diff > 0 else 0.0
                pub_robot.send(obj=get_qmsg_from_qpos(qpos=q_use,q_msg_layout=q_msg_layout))
            else:
                robot_pub_hz_actual = 0.0
            tmr_robot.end()

        if tmr_render.do_run():
            env.plot_global_coordinate_axes()
            env.plot_links_between_bodies(r=0.001)
            for marker in markers.values():
                marker.draw()
            env.viewer_text_overlay("Command","robot %s | pub %.1fHz"%("ON" if on_flag else "OFF",robot_pub_hz_actual),loc="top left")
            env.viewer_text_overlay("Marker","\n".join("%s: %s"%(hand_labels[hand_type],markers[hand_type].get_status()) for hand_type in hand_types),loc="top left")
            env.viewer_text_overlay("IK",make_sg2_ik_overlay_text(collision_ik_result),loc="top left")
            env.viewer_text_overlay("Collision",make_sg2_collision_overlay_text(collision_ik_result),loc="top left")
            env.viewer_text_overlay("Gripper"," | ".join("%s %.2f"%(hand_labels[hand_type],gripper_q_by_hand[hand_type]) for hand_type in hand_types),loc="top left")
            env.render()

            send_render_qpos(
                pub_render          = pub_render,
                qpos                = q_use,
                joint_names         = command_joint_names,
                q_msg_layout        = q_msg_layout,
                msg_type            = "real_marker_wrist_ffw_sg2",
                on_flag             = on_flag,
                robot_hz            = robot_hz,
                render_hz           = render_hz,
                robot_pub_hz_actual = robot_pub_hz_actual,
                extra               = {
                    "raw_robot_command_on" : on_flag,
                    "attached"             : {hand_type:True for hand_type in hand_types},
                    "attach_phase"         : {hand_type:"marker" for hand_type in hand_types},
                    "attach_progress"      : {hand_type:1.0 for hand_type in hand_types},
                    "quest_ok"             : {hand_type:False for hand_type in hand_types},
                    "quest_all_ok"         : False,
                    "collision_ik_mode"    : collision_ik_result.get("mode","idle"),
                    "min_signed_dist"      : float(collision_ik_result.get("min_signed_dist",np.nan)),
                    "safe_margin"          : float(collision_ik_result.get("safe_margin",0.05)),
                    "T_quest_target"       : copy_hand_T_dict(wrist_targets_by_hand),
                    "T_sg2_wrist"          : copy_hand_T_dict(T_sg2_wrist),
                    "active_arm_ik_targets": render_arm_ik_targets,
                    "arm_ik_err_best"      : {
                        hand_type:None
                        if arm_ik_info_by_hand[hand_type] is None
                        else float(arm_ik_info_by_hand[hand_type]["ik_err_best"])
                        for hand_type in hand_types
                    },
                    "gripper_q"            : dict(gripper_q_by_hand),
                },)
            tmr_render.end()

except KeyboardInterrupt:
    print("Interrupted.")
finally:
    pub_render.close()
    pub_robot.close()
    gripper_slider.close()
    ctrl_radio.close()
    if getattr(env,"final_rgb_img",None) is not None:
        imshow(env.final_rgb_img,fig_w=6,title="Final scene",title_fs=8)
    env.close_viewer()
    print("Done.")

[Marker IK] Initialized with [30.0]HZ. MAX_SEC:[inf]sec.
[Robot Pub] Initialized with [100.0]HZ. MAX_SEC:[inf]sec.
[Render Pub] Initialized with [20.0]HZ. MAX_SEC:[inf]sec.
[Marker IK] Start ([30.0]HZ. MAX_SEC:[inf]sec).
[Robot Pub] Start ([100.0]HZ. MAX_SEC:[inf]sec).
[Render Pub] Start ([20.0]HZ. MAX_SEC:[inf]sec).
[Render Pub][1] run delayed! T:[50.0]ms run:[179.1]ms]
[Marker IK][2][0.2s] delayed! T:[33.3]ms interval:[183.2]ms prev_run:[3.9]ms.
[Robot Pub][2][0.2s] delayed! T:[10.0]ms interval:[181.8]ms prev_run:[0.0]ms.
[Render Pub][2][0.2s] delayed! T:[50.0]ms interval:[181.8]ms prev_run:[179.1]ms.
[Render Pub][2] run delayed! T:[50.0]ms run:[116.9]ms]
[Marker IK][3][0.3s] delayed! T:[33.3]ms interval:[119.6]ms prev_run:[2.5]ms.
[Robot Pub][3][0.3s] delayed! T:[10.0]ms interval:[119.9]ms prev_run:[0.0]ms.
[Render Pub][3][0.3s] delayed! T:[50.0]ms interval:[119.9]ms prev_run:[116.9]ms.
[Render Pub][3] run delayed! T:[50.0]ms run:[61.1]ms]
[Marker IK][4][0.4s] delayed! T:[33.3]ms in